In [1]:
import pandas as pd
import os 
import pandas as pd
import xarray as xr
import numpy as np
from scipy.spatial import cKDTree

#path = os.getcwd()+'/data/FLUXNET/FLX_AA-Flx_BIF_ALL_20200501/'
path = os.getcwd()+'/data/FLUXNET_Data_Shuttle/'

# Specify the file path
file_path = path+'fluxnet_shuttle_snapshot_20260428T165716_selectedsites.csv'

data = pd.read_csv(file_path)

In [2]:
# Get unique SITE_ID values
site_ids = data['site_id'].unique()

# Define the metadata variables
metadata_variables = ['site_name','location_lat', 'location_long',
                      'igbp', 'product_citation']

# Convert the metadata list to a DataFrame
metadata_table = pd.DataFrame(data)

# Display the metadata table
print(metadata_table)

# Rename the columns
new_column_names = {
    'site_id': 'Site ID',
    'site_name': 'Name',
    'location_lat': 'Lat',
    'location_long': 'Long',
    'igbp': 'Veg',
    'product_citation': 'citation'
}

metadata_table.rename(columns=new_column_names, inplace=True)


      data_hub site_id                   site_name  location_lat  \
0    AmeriFlux  AR-CCa  Carlos Casares agriculture    -35.621000   
1    AmeriFlux  AR-CCg    Carlos Casares grassland    -35.924400   
2         ICOS  AT-Mmg                     Mieming     47.316563   
3         ICOS  AT-Neu                    Neustift     47.116318   
4         ICOS  AT-Nsd                    Neusiedl     47.769140   
..         ...     ...                         ...           ...   
457       ICOS  ZA-BfK            Benfontien Karoo    -28.856483   
458       ICOS  ZA-BfS          Benfontien Savanna    -28.890600   
459       ICOS  ZA-Jks                 Jonkershoek    -33.990290   
460       ICOS  ZA-Spk                   Spioenkop    -28.704080   
461       ICOS  ZA-Uby            Umhlabuyalingana    -27.395200   

     location_long igbp                                            network  \
0       -61.318100  CRO                                          AmeriFlux   
1       -61.185500  GRA    

In [3]:
# Make sure Lat and Long are numeric
metadata_table['Lat'] = pd.to_numeric(metadata_table['Lat'], errors='coerce')
metadata_table['Long'] = pd.to_numeric(metadata_table['Long'], errors='coerce')

# === 2. Load Climate Map ===
nc_path = "data/Support/koppen_geiger_0p1.nc"
ds = xr.open_dataset(nc_path)
lat = ds['lat'].values
lon = ds['lon'].values
kg_class = ds['kg_class'].values
kg_confidence = ds['kg_confidence'].values

# === 3. Build KDTree for fast lat/lon matching ===
# Use indexing='ij' so row and col match correctly
lat_grid, lon_grid = np.meshgrid(lat, lon, indexing='ij')
points = np.column_stack((lat_grid.ravel(), lon_grid.ravel()))
tree = cKDTree(points)

# === 4. Extract climate data for each site ===
def extract_climate_for_site(lat_val, lon_val):
    if np.isnan(lat_val) or np.isnan(lon_val):
        return np.nan, np.nan
    _, idx = tree.query([lat_val, lon_val], k=1)
    row, col = divmod(idx, len(lon))  # This now works correctly
    return float(kg_class[row, col]), float(kg_confidence[row, col])

metadata_table[['kg_class', 'kg_confidence']] = metadata_table.apply(
    lambda row: extract_climate_for_site(row['Lat'], row['Long']),
    axis=1, result_type='expand'
)

# Example mapping (fill this with the full key if needed)
koppen_labels = {
    1: "Af",   2: "Am",   3: "Aw",
    4: "BWh",  5: "BWk",  6: "BSh",  7: "BSk",
    8: "Csa",  9: "Csb", 10: "Csc",
    11: "Cwa", 12: "Cwb", 13: "Cwc",
    14: "Cfa", 15: "Cfb", 16: "Cfc",
    17: "Dsa", 18: "Dsb", 19: "Dsc",
    20: "Dsd", 21: "Dwa", 22: "Dwb",
    23: "Dwc", 24: "Dwd", 25: "Dfa",
    26: "Dfb", 27: "Dfc", 28: "Dfd",
    29: "ET",  30: "EF"
}

metadata_table["kg_label"] = metadata_table["kg_class"].map(koppen_labels)

# === 5. Save or return the result ===
metadata_table.to_csv(path+'complete_fluxnet_data_shuttle_metadata_table.csv', index=False)